In [ ]:
# -*- coding: utf-8 -*-
#  Copyright 2024 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#  Authored by:    Rasmia Kulan (UKRI-STFC)
#                  Laura Murgatroyd (UKRI-STFC)

# Aluminium Cylinder - Demonstration of Effect of Reduced Projections / Exposure on FBP Reconstructions

This exercise demonstrates how reducing the number of projections and exposure time affects the reconstruction of the data achieved with Filtered Backprojection (FBP).

This is particularly relevant for neutron tomography experiments at IMAT because due to the low flux the experiments take a long time.

This notebook will:
- Load and investigate neutron tomography data with different exposure times and number of projections
- Apply Filtered Back Projection (FBP) Reconstruction on 3 slices of interest.
- Compare the FBP to study the effect of reducing angular sampling and exposure time, using the following metrics:
    - Signal to Noise Ratio (SNR) of ROIs of rods in the sample.
    - Contrast to Noise Ratio (CNR) between rods and aluminium cylinder.
    - L2-Norm Error metric to all the three slices and three different ROIs.

Imports:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cil.utilities.display import show1D, show2D
from cil.processors import Slicer
from cil.optimisation.functions import L2NormSquared
from cil.processors import Slicer


from cil.plugins.astra import FBP as FBP_ASTRA

### Load Data

This notebook will compare the following data:

| Exposure Time (s) | Angles | Total Experiment Time (h) | Fraction of Longest Experiment|
|-------------------|--------|----------------------------|----------------------|
| 60                | 840    | 14.0                       | 1                    |
| 60                | 420    | 7.0                        | 1/2                  |
| 60                | 210    | 3.5                        | 1/4                  |
| 60                | 105    | 1.75                       | 1/8                  |
| 30                | 210    | 1.75                       | 1/8                  |
| 15                | 420    | 1.75                       | 1/8                  |
| 7.5               | 840    | 1.75                       | 1/8                  |

However, it is important to note that the total experiment time does not include additional time required for the experiment including rotating the sample, read out from the camera, etc., which significantly increases the total time.

Note, as well as comparing the quality of the reconstruction as the total experiment time is reduced, for the total experiment time of 1/8 of the ideal length, we compare different combinations of reduction of angles and exposure times.



Here we use a method  from `data_io` to load the data listed in the above table. The data we load here has already been pre-processed in Mantid Imaging.

In [ ]:
from data_io.alum_cyl_io import read_processed_data

In [ ]:
exp_1min_angles_840 = read_processed_data(exposure_time=60, num_angles=840)
exp_1min_angles_420 = read_processed_data(exposure_time=60, num_angles=420)
exp_1min_angles_210 = read_processed_data(exposure_time=60, num_angles=210)
exp_1min_angles_105 = read_processed_data(exposure_time=60, num_angles=105)
exp_30s_angles_210 = read_processed_data(exposure_time=30, num_angles=210)
exp_15s_angles_420 = read_processed_data(exposure_time=15, num_angles=420)
exp_7_5s_angles_840 = read_processed_data(exposure_time=7.5, num_angles=840)


processed_data_dict = {'exp_1min_840': exp_1min_angles_840, 'exp_1min_420': exp_1min_angles_420,
    'exp_1min_210': exp_1min_angles_210, 'exp_1min_105': exp_1min_angles_105,
    'exp_30s_210': exp_30s_angles_210, 'exp_15s_420': exp_15s_angles_420, 'exp_7.5s_840': exp_7_5s_angles_840}

If you have any trouble loading the data in the above cell, please see the start of the [Angles_vs_Exposure_01_Introduction_to_Aluminium_Cylinder.ipynb](Angles_vs_Exposure_01_Introduction_to_Aluminium_Cylinder.ipynb) notebook.

### Reconstruct with FBP from ASTRA

For each dataset and for each slice of interest, the data is reconstructed using FBP from Astra:

In [ ]:
sliceA_slicer = Slicer(roi={'vertical': (245, 246)})
sliceB_slicer = Slicer(roi={'vertical': (623, 624)})
sliceC_slicer = Slicer(roi={'vertical': (987, 988)})

In [ ]:
sliced_recon_dict = {key:[] for key in processed_data_dict.keys()}

for key, data in processed_data_dict.items():

    data.reorder('astra')
    recon = FBP_ASTRA(data.geometry.get_ImageGeometry(), data.geometry)(data)
    reconA = sliceA_slicer(recon)
    reconB = sliceB_slicer(recon)
    reconC = sliceC_slicer(recon)
    sliced_recon_dict[key] = [reconA, reconB, reconC]


In [ ]:
all_fbp_grid = [sliced_recon_dict[key][i] for key in sliced_recon_dict.keys() for i in range(3)]

# find highest max value for display purposes:
min=0
max=0
for x in all_fbp_grid:
    if x.max() > max:
        max = x.max()

labels = [
    'exp=1min, proj=840, time=14h',
    'exp=1min, proj=420, time=7h',
    'exp=1min, proj=210, time=3.5h',
    'exp=1min, proj=105, time=1.75h',
    'exp=30s, proj=210, time=1.75h',
    'exp=15s, proj=420, time=1.75h',
    'exp=7.5s, proj=840, time=1.75h',
]

all_labels_grid = []
for label in labels:
    all_labels_grid.append('Slice A FBP [' + label + ']')
    all_labels_grid.append('Slice B FBP [' + label + ']')
    all_labels_grid.append('Slice C FBP [' + label + ']')


show2D(all_fbp_grid, all_labels_grid, cmap='inferno', fix_range=(min, max), num_cols=3);

It's not easy to compare in that grid, but it displays all of the reconstructions we'll be evaluating in this notebook!
An example of a closer look at two reconstructions is shown below, looking at a 60s compared to a 7.5s exposure time:

In [ ]:
show2D([sliced_recon_dict['exp_1min_840'][0],sliced_recon_dict['exp_7.5s_840'][0]],  [all_labels_grid[0], all_labels_grid[-3]], cmap='inferno', fix_range=(min, max), num_cols=3);

In [ ]:
show2D([sliced_recon_dict['exp_1min_840'][1],sliced_recon_dict['exp_7.5s_840'][1]],  [all_labels_grid[1], all_labels_grid[-2]], cmap='inferno', fix_range=(min, max), num_cols=3);

In [ ]:
show2D([sliced_recon_dict['exp_1min_840'][2],sliced_recon_dict['exp_7.5s_840'][2]],  [all_labels_grid[2], all_labels_grid[-1]], cmap='inferno', fix_range=(min, max), num_cols=3);

We see a lot more noise with the lower exposure time and it looks like it will be harder to distinguish smaller features.

### Evaluation of FBP Reconstruction with Metrics

We'll be looking at ROIs relating to rods in the cylinder, which we'll refer to with numbered labels as shown in this image:

<img src="images/cylinder_regions.png" width=800 align="center">

#### Contrast to Noise Ratio

In this section we will focus on slice B and evaluate the Contrast to Noise Ratio between the rods and the aluminium cylinder.

For each rod in slice B we first create an ROI which includes the rod and some aluminium cylinder background. We create this for all datasets but just display for the 60s exposure, 840 angles dataset:

In [ ]:
def crop_rod_B(rec):
    """Crop four rods from a reconstruction slice."""
    rod3 = Slicer(roi={'horizontal_x': (280, 350), 'horizontal_y': (135, 200)})(rec)
    rod4 = Slicer(roi={'horizontal_x': (165, 235), 'horizontal_y': (285, 350)})(rec)
    rod1 = Slicer(roi={'horizontal_x': (315, 385), 'horizontal_y': (395, 460)})(rec)
    rod2 = Slicer(roi={'horizontal_x': (425, 495), 'horizontal_y': (247, 312)})(rec)
    return [rod1, rod2, rod3, rod4]

labels_slice_B_rods = ["Slice B " + label for label in labels]

slice_B_data = {}

for i, data_label in enumerate(sliced_recon_dict.keys()):
    rec = sliced_recon_dict[data_label][1]
    rods = crop_rod_B(rec)
    current_labels = [f"{labels_slice_B_rods[i]} Rod 1", "Rod 2", "Rod 3", "Rod 4"]

    slice_B_data[data_label] = {"rods": rods, "labels": current_labels}

all_rods_B = [rod for entry in slice_B_data.values() for rod in entry["rods"]]
all_labels_B = [label for entry in slice_B_data.values() for label in entry["labels"]]

show2D(all_rods_B[0:4], num_cols=4, cmap="inferno", fix_range=(min, max));

One way we could examine the contrast is using a line profile through this ROI. We display this for rod 1:

In [ ]:
linenumy = 30

rods = [slice_B_data[x]["rods"][0] for x in slice_B_data.keys()]

print(labels)

show1D(rods,
        slice_list=[('horizontal_y',linenumy)],
        title='Line profile through horizontal_y=30 Slice B Rod 1',
        dataset_labels=labels,
        line_colours=['blue','red','green','orange', 'purple', 'cyan', 'grey'],
        line_styles=['solid']*7,
        size=(22, 15))

In particular we see that the 1.75hr acquisitions are particularly noisy.

Next we create ROIs with only the material inside each rod, plus an additional ROI of the aluminium cylinder - 'background'.

In [ ]:
data_labels = [ x for x in slice_B_data.keys()]

for data_label in data_labels:
    slice_B_data[data_label]['rods_inner'] = []
    for i in range(4):
        data = slice_B_data[data_label]["rods"][i]
        roi_img = Slicer(roi={'horizontal_x': (25,45), 'horizontal_y': (25,45)})(data)
        slice_B_data[data_label]['rods_inner'].append(roi_img)

In [ ]:
for data_label in data_labels:
    rec = sliced_recon_dict[data_label][1]
    bkg = Slicer(roi = {'horizontal_x': (300, 320), 'horizontal_y': (400, 420)})(rec)
    label = "Background"
    slice_B_data[data_label]["rods_inner"].append(bkg)
    slice_B_data[data_label]["labels"].append(label)

In [ ]:
show2D(slice_B_data['exp_1min_840']['rods_inner'], slice_B_data['exp_1min_840']['labels'], fix_range=(min,max), cmap='inferno', num_cols=5);

# To show all regions:
# show2D([item for sublist in [slice_B_data[x]['rods_inner'] for x in data_labels] for item in sublist], 
#        [item for sublist in [slice_B_data[x]['labels'] for x in data_labels] for item in sublist],
#        fix_range=(min,max), cmap='inferno', num_cols=5)

##### Contrast to Noise Ratio (CNR)
This metric assesses the contrast between the background and the ROI.
&nbsp;

Method:
&nbsp;

$$
\text{CNR} = \frac{\text{Mean of ROI} - \text{Mean of Background}}{\text{Standard Deviation of Background}}
$$
&nbsp;

We will compare an ROI within each rod with an ROI in the aluminium cylinder - 'background', in slice B.

In [ ]:
def calculate_cnr(data, bkg_data):
    mean_roi = np.mean(data.as_array())
    mean_bkg = np.mean(bkg_data.as_array())
    std_bkg = np.std(bkg_data.as_array())
    cnr = (mean_roi - mean_bkg) / std_bkg
    return cnr

In [ ]:
for data_label in data_labels:
    slice_B_data[data_label]['cnrs'] = []
    for i in range(4):
        data = slice_B_data[data_label]["rods_inner"][i]
        bkg_data = slice_B_data[data_label]["rods_inner"][4]
        slice_B_data[data_label]['cnrs'].append(calculate_cnr(data, bkg_data))

In [ ]:
x = np.arange(len(labels))[0:4]  # x locations for the groups

width = 0.2  # bar width

plt.figure(figsize=(10, 6))

for i in range(4):
    cnrs = [slice_B_data[x_label]['cnrs'][i] for x_label in list(slice_B_data.keys())[0:4]]
    plt.bar(x + i * width - 1.5*width, cnrs, width, label=f'Rod {i+1}')

plt.xticks(x, labels[0:4], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('CNR')
plt.title('CNR between rods and background for exposure 1min and varied number of Projections')
plt.legend()
plt.tight_layout()
plt.minorticks_on()
plt.grid(axis='y', which='major', linestyle='-', alpha=0.5)
plt.grid(axis='y', which='minor', linestyle='--', alpha=0.25)
plt.show()

x = np.arange(len(labels))[3:7]  # x locations for the groups

width = 0.2  # bar width

plt.figure(figsize=(10, 6))

for i in range(4):
    cnrs = [slice_B_data[x_label]['cnrs'][i] for x_label in list(slice_B_data.keys())[3:7]]
    plt.bar(x + i * width - 1.5*width, cnrs, width, label=f'Rod {i+1}')

plt.xticks(x, labels[3:7], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('CNR')
plt.title('CNR between rods and background for total experiment time 1.75h')
plt.legend()
plt.tight_layout()
plt.minorticks_on()
plt.tick_params(axis='x', which='minor', bottom=False)
plt.grid(axis='y', which='major', linestyle='-', alpha=0.5)
plt.grid(axis='y', which='minor', linestyle='--', alpha=0.25)
plt.show()


As expected, we see the CNR decreasing as the total experiment time increases.

For the 1.75h experiment time, the combination with the best contrast to noise ratio is exp=30s and proj=210. 

However, the results are not much different to the exp=15s and proj=420. Note these are respectively 1/2 longest exposure, 1/4 highest projection number and 1/4 longest exposure, 1/2 highest projection number.

### Signal to Noise Ratio (SNR)
This is the ratio of signal compared to noise in the region. 
&nbsp;

Method:
&nbsp;

$$
\text{SNR} = \frac{\text{Mean of ROI}}{\text{Standard Deviation of ROI}}
$$
&nbsp;

We will be using the Slice B dataset to apply SNR of different angles/exposures through the cylinder background and the four rods.


In [ ]:
def calculate_snr(data):
    mean_roi = np.mean(data.as_array())
    std_roi = np.std(data.as_array())
    snr = mean_roi / std_roi
    return snr

data_labels = [ x for x in slice_B_data.keys()]

for data_label in data_labels:
    slice_B_data[data_label]['snrs'] = []
    for i in range(5):
        data = slice_B_data[data_label]["rods_inner"][i]
        slice_B_data[data_label]['snrs'].append(calculate_snr(data))

In [ ]:
def apply_plot_settings():
    plt.legend()
    plt.tight_layout()
    plt.minorticks_on()
    plt.tick_params(axis='x', which='minor', bottom=False)
    plt.grid(axis='y', which='major', linestyle='-', alpha=0.5)
    plt.grid(axis='y', which='minor', linestyle='--', alpha=0.25)

In [ ]:
x = np.arange(len(labels))[0:4]  # x locations for the groups

width = 1/6  # bar width

plt.figure(figsize=(10, 6))

for i in range(5):
    snrs = [slice_B_data[x]['snrs'][i] for x in list(slice_B_data.keys())[0:4]]
    #plt.plot(labels, snrs, marker='o', label=f'SNR of Rod {i+1}' if i<4 else 'SNR of Cylinder')
    plt.bar(x + i * width - 2*width, snrs, width, label=f'Rod {i+1}' if i<4 else 'Cylinder')

plt.xticks(x, labels[0:4], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('SNR')
plt.title('SNR of ROIs for exposure 1min and varied number of Projections')
apply_plot_settings()
plt.show()


In [ ]:
x = np.arange(len(labels))[3:7]  # x locations for the groups

width = 1/6  # bar width

plt.figure(figsize=(10, 6))

for i in range(5):
    snrs = [slice_B_data[x]['snrs'][i] for x in list(slice_B_data.keys())[3:7]]
    #plt.plot(labels, snrs, marker='o', label=f'SNR of Rod {i+1}' if i<4 else 'SNR of Cylinder')
    plt.bar(x + i * width - 2*width, snrs, width, label=f'Rod {i+1}' if i<4 else 'Cylinder')

plt.xticks(x, labels[3:7], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('SNR')
plt.title('SNR of ROIs for total experiment time 1.75h')
apply_plot_settings()
plt.show()


As expected, the signal to noise ratio decreases for all ROIs as we reduce the total acquisition time.

For the 1.75h experiment time, the combination with the best SNR is exp=30s and proj=210.

However, the results are not much different to the exp=15s and proj=420. Note these are respectively 1/2 longest exposure, 1/4 highest projection number and 1/4 longest exposure, 1/2 highest projection number.

THE SNR is best when we have a balance between reduction in number of projections and exposure time.

### L2Norm Error (L2N-E)
This metric assess the difference in pixel values between a reconstruction and the ground truth.
&nbsp;

Method:
&nbsp;

$$
\text{L2N-E} = \frac{\| \text{Recon} - \text{Ground Truth} \|_2}{\| \text{Ground Truth} \|_2}
$$
&nbsp;

We will be applying this metric on datasets of Slices A, B and C to compare the L2N-E of different angles/exposures with the ground truth.

In [ ]:
def calculate_L2Norm_error(ground_truth, rec):
    """Compute normalized L2 error between ground truth and reconstruction."""
    return np.sqrt(L2NormSquared(b=ground_truth)(rec) / L2NormSquared()(ground_truth))

In [ ]:
# --- Define slice labels and corresponding ground truths ---
slice_indices = {'A': 0, 'B': 1, 'C': 2}
ground_truths = {k: sliced_recon_dict['exp_1min_840'][v] for k, v in slice_indices.items()}

# --- Initialize structure for storing errors ---
l2_errors = {k: [] for k in slice_indices.keys()}

# --- Compute L2Norm errors for each configuration ---
for recs in sliced_recon_dict.values():
    for slice_key, idx in slice_indices.items():
        l2_errors[slice_key].append(
            calculate_L2Norm_error(ground_truths[slice_key], recs[idx])
        )



In [ ]:
x = np.arange(len(labels))[1:4]  # x locations for the groups

width = 0.25

plt.figure(figsize=(10, 6))

i=0
for slice_key, errors in l2_errors.items():
    plt.bar(x+ i * width - width, errors[1:4], width, label=f' Slice {slice_key}')
    i+=1

plt.xticks(x, labels[1:4], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('L2Norm Error')
plt.title('L2Norm Error for exposure 1min and varied number of projections')
apply_plot_settings()
plt.show()


In [ ]:
x = np.arange(len(labels))[3:7]
plt.figure(figsize=(10, 6))

i=0
for slice_key, errors in l2_errors.items():
    plt.bar(x+ i * width - width, errors[3:7], width, label=f' Slice {slice_key}')
    i+=1

plt.xticks(x, labels[3:7], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('L2Norm Error')
plt.title('L2Norm Error for total experiment time 1.75h')
apply_plot_settings()
plt.show()


As expexted, we see the error increasing as the total experiment time increases.
For the 1.75h experiment time, the combination with the lowest error is exp=30s and proj=210

However, the results are not much different to the exp=15s and proj=420. Note these are respectively 1/2 longest exposure, 1/4 highest projection number and 1/4 longest exposure, 1/2 highest projection number.

This section takes three different ROI from the three slices and applies the L2Norm Error metric. This is done to study the behaviour of the metric with respect to object size.
First we will display the 3 different ROIs in the 1min exposure data vs the 7.5s exposure data, both with 840 angles. Then we will display the L2Norm calculation results.

In [ ]:
slicer_rod = Slicer(roi={'horizontal_x': (345, 415), 'horizontal_y': (400, 470)})
slicer_boundary = Slicer(roi={'horizontal_x': (160, 230), 'horizontal_y': (60, 130)})
slicer_beads = Slicer(roi={'horizontal_x': (335, 405), 'horizontal_y': (385, 455)})


slice_A_data,  slice_C_data = {}, {}

for key in data_labels:
    slice_A_data[key] = {"L2NormROI": slicer_rod(sliced_recon_dict[key][0])}
    slice_B_data[key]["L2NormROI"] =  slicer_boundary(sliced_recon_dict[key][1])
    slice_C_data[key] = {"L2NormROI": slicer_beads(sliced_recon_dict[key][2])}

# --- Select ground truth samples ---
ground_truth_a = slice_A_data["exp_1min_840"]["L2NormROI"]
ground_truth_b = slice_B_data["exp_1min_840"]["L2NormROI"]
ground_truth_c = slice_C_data["exp_1min_840"]["L2NormROI"]

show2D([ground_truth_a, ground_truth_b, ground_truth_c], fix_range=(min,max), cmap='inferno', num_cols=3, title=["Slice A Rod [exp=1min, angles=840]","Slice B Boundary [exp=1min, angles=840]","Slice C Beads [exp=1min, angles=840]"]);

# Show the images for the lowest quality data
show2D([slice_A_data['exp_7.5s_840']['L2NormROI'],
        slice_B_data['exp_7.5s_840']['L2NormROI'],
        slice_C_data['exp_7.5s_840']['L2NormROI']],
       ['Slice A Rod [exp=7.5s, angles=840]',
        'Slice B Boundary [exp=7.5s, angles=840]',
        'Slice C Beads [exp=7.5s, angles=840]'],
       fix_range=(min,max), cmap='inferno', num_cols=3);

In [ ]:
# --- Compute L2Norm errors for all datasets ---
for key in data_labels:
    slice_A_data[key]['L2Norm'] = calculate_L2Norm_error(ground_truth_a, slice_A_data[key]['L2NormROI'])
    slice_B_data[key]['L2Norm'] = calculate_L2Norm_error(ground_truth_b, slice_B_data[key]['L2NormROI'])
    slice_C_data[key]['L2Norm'] = calculate_L2Norm_error(ground_truth_c, slice_C_data[key]['L2NormROI'])

# --- Prepare data for plotting ---
slice_A = [slice_A_data[key]['L2Norm'] for key in data_labels]
slice_B = [slice_B_data[key]['L2Norm'] for key in data_labels]
slice_C = [slice_C_data[key]['L2Norm'] for key in data_labels]

In [ ]:
x = np.arange(len(labels))[1:4]

width = 0.25

plt.figure(figsize=(10, 6))
plt.bar(x - width, slice_A[1:4], width, label=f' Slice A')
plt.bar(x, slice_B[1:4], width, label=f' Slice B')
plt.bar(x + width, slice_C[1:4], width, label=f' Slice C')

plt.xticks(x, labels[1:4], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('L2Norm Error')
plt.title('L2Norm Error of ROIs for exposure 1min and varied number of projections')
apply_plot_settings()
plt.show()

x = np.arange(len(labels))[3:7]

plt.figure(figsize=(10, 6))
plt.bar(x - width, slice_A[3:7], width, label=f' Slice A')
plt.bar(x, slice_B[3:7], width, label=f' Slice B')
plt.bar(x + width, slice_C[3:7], width, label=f' Slice C')

plt.xticks(x, labels[3:7], fontsize=8)
plt.xlabel('Exposure/Angle Configuration')
plt.ylabel('L2Norm Error')
plt.title('L2Norm Error of ROIs for exposure 1min and varied number of projections')
apply_plot_settings()
plt.show()



As expected, we see the error increasing as the total experiment time increases.
For the 1.75h experiment time, the combination with the lowest error is exp=30s and proj=210.
However, the results are not much different to the exp=15s and proj=420. Note these are respectively 1/2 longest exposure, 1/4 highest projection number and 1/4 longest exposure, 1/2 highest projection number.

## Conclusion



Across all 3 metrics, we see the best outcome with the FBP reconstruction with experiment time 1.75h was with exp=30s and proj=210.
However, the results are not much different to the exp=15s and proj=420.
This indicates that some balance between reduction in exposure time and number of projections can be found to optimise image quality within a fixed experiment time. 
This has been found to be preferable to only reducing one of exposue time or number of projections.